# 02 -- SMCD design walkthrough

Narrative only (`project.md` SS9). Runs Algorithm 1 (SMCD) on every one of this
project's 6 problem instances and prints the resulting design card -- the same
deterministic, no-search construction every `configs/exp/*.yaml` run in
`results/runs/` actually used (`qapinn.runner.PROBLEM_INSTANCES`, the single source of
truth every sweep config draws from).

In [ ]:
from qapinn.runner import PROBLEM_INSTANCES
from qapinn.config import load_config
import qapinn.pdes as pdes_pkg
from qapinn.smcd.design import smcd

print(f"{len(PROBLEM_INSTANCES)} problem instances:")
for label, pde_yaml, overrides in PROBLEM_INSTANCES:
    print(f"  {label:16s} -> {pde_yaml} {overrides or ''}")

## Design card per instance

`load_config` resolves each instance's `configs/pde/<name>.yaml` plus its
instance-specific overrides into a real `ExpConfig` -- the exact object every training
run uses -- and `qapinn.pdes.build(cfg.pde)` (the same call `train/loop.py` makes)
constructs the actual `PDE` object `smcd()` designs against.

In [ ]:
rows = []
for label, pde_yaml, overrides in PROBLEM_INSTANCES:
    cfg = load_config(pde_yaml, "q_serial", overrides=overrides)
    pde = pdes_pkg.build(cfg.pde)
    card = smcd(pde)
    rows.append((label, card.n_qubits, card.n_layers, card.entangler, card.coverage, card.predicted_benefit))

print(f"{'problem':16s} {'n_qubits':>8s} {'n_layers':>8s} {'entangler':>10s} {'coverage':>9s} {'pred_benefit':>13s}")
for label, n, L, ent, cov, pb in rows:
    print(f"{label:16s} {n:8d} {L:8d} {ent:>10s} {cov:9.3f} {pb:13.4f}")

## Reading this table

`n_qubits`/`n_layers` are Algorithm 1's deterministic output for each PDE's target
spectrum -- nothing here is architecture-searched or hand-tuned per problem.
`predicted_benefit` is SMCD's a-priori heuristic (Section~5 of the paper), computed
before any training happens; `paper/figures/decision_map.pdf` plots this against what
was actually measured (`FINDINGS.md`).